# Prompt Quality Scoring (Pre-Inference Heuristics)
Evaluate the structural quality of a prompt before sending it to Claude.
Seven rule-based dimensions — no API calls required for scoring.

In [ ]:
%%capture
%pip install anthropic

In [ ]:
import os
import re
import anthropic

client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))
MODEL_NAME = "claude-haiku-4-5"

## Step 1: Define Scoring Dimensions
Each dimension is detected with a regex heuristic. No LLM call needed.

In [ ]:
# --- Dimension 1: Role / Persona Definition ---
ROLE_PATTERNS = [
    r"(?i)^you are\b",
    r"(?i)^act as\b",
    r"(?i)^assume the role\b",
    r"(?i)^you're a\b",
    r"(?i)<system>",
]

def has_role_definition(prompt: str) -> bool:
    return any(re.search(p, prompt.strip()) for p in ROLE_PATTERNS)


# --- Dimension 2: Instruction Clarity ---
IMPERATIVE_VERBS = [
    "summarize", "list", "write", "explain", "analyze", "compare",
    "translate", "classify", "extract", "generate", "create", "describe",
    "identify", "evaluate", "suggest", "provide", "return",
]

def instruction_clarity_score(prompt: str) -> float:
    sentences = [s.strip() for s in re.split(r"[.!?]", prompt) if s.strip()]
    if not sentences:
        return 0.0
    imperative_count = sum(
        1 for s in sentences
        if any(s.lower().startswith(v) for v in IMPERATIVE_VERBS)
    )
    return min(imperative_count / max(len(sentences), 1), 1.0)


# --- Dimension 3: Output Format Specification ---
FORMAT_KEYWORDS = [
    r"(?i)\bjson\b", r"(?i)\bxml\b", r"(?i)\bmarkdown\b",
    r"(?i)\bbulleted? list\b", r"(?i)\bnumbered list\b",
    r"(?i)\btable\b", r"(?i)\bcsv\b", r"(?i)\bplain text\b",
    r"(?i)return (?:a |an |the )?(?:json|list|dict|string|number)",
    r"(?i)format (?:your |the )?(?:response|output|answer) as",
    r"(?i)respond (?:only )?(?:in|with)\b",
    r"(?i)<output_format>",
]

def has_output_format(prompt: str) -> bool:
    return any(re.search(p, prompt) for p in FORMAT_KEYWORDS)


# --- Dimension 4: Few-Shot Example Inclusion ---
FEW_SHOT_PATTERNS = [
    r"(?i)<example>",
    r"(?i)for example:",
    r"(?i)\bexample\s*\d+\s*[:\-]",
    r"(?i)\binput\s*:\s*.+\n.+output\s*:",
    r"(?i)here(?:'s| is) an example",
    r"(?i)\be\.g\.\b",
]

def has_few_shot_examples(prompt: str) -> bool:
    return any(re.search(p, prompt) for p in FEW_SHOT_PATTERNS)


# --- Dimension 5: Constraint Specification ---
CONSTRAINT_PATTERNS = [
    r"(?i)\bat most\b",
    r"(?i)\bno more than\b",
    r"(?i)\bin (?:a |an )?(?:formal|professional|casual|concise|brief|friendly)\b",
    r"(?i)\bmust not\b",
    r"(?i)\bdo not\b",
    r"(?i)\bonly (?:use|include|mention|return)\b",
    r"(?i)\blimit(?: your| the)?\b.{0,30}\bto\b",
    r"(?i)\bwithin \d+ (?:words?|sentences?|characters?|lines?)\b",
    r"(?i)<constraints?>",
]

def has_constraints(prompt: str) -> bool:
    return any(re.search(p, prompt) for p in CONSTRAINT_PATTERNS)


# --- Dimension 6: Context / Background Completeness ---
CONTEXT_PATTERNS = [
    r"(?i)<context>",
    r"(?i)\bthe following\b.{0,60}\b(?:text|document|article|passage|content)\b",
    r"(?i)\bgiven (?:the|this|that)\b",
    r"(?i)\bbelow is\b",
    r"(?i)\bhere is (?:the|a|some)\b",
    r"(?i)\bbackground\s*:\s*",
]

def has_context(prompt: str) -> bool:
    return any(re.search(p, prompt) for p in CONTEXT_PATTERNS)


# --- Dimension 7: Task Decomposition ---
DECOMP_PATTERNS = [
    r"(?m)^\s*(?:step\s*)?\d+[\.)\]\s+\S",
    r"(?i)first[,\s].{0,60}then[,\s].{0,60}finally",
    r"(?i)<steps?>",
    r"(?i)\bfollowing steps\b",
]

def has_task_decomposition(prompt: str) -> bool:
    return any(re.search(p, prompt) for p in DECOMP_PATTERNS)

In [ ]:
# --- Composite scorer ---
WEIGHTS = {
    "role_definition":      0.20,
    "instruction_clarity":  0.20,
    "output_format":        0.20,
    "few_shot_examples":    0.15,
    "constraints":          0.10,
    "context_completeness": 0.10,
    "task_decomposition":   0.05,
}

def score_prompt(prompt: str) -> dict:
    raw = {
        "role_definition":      float(has_role_definition(prompt)),
        "instruction_clarity":  instruction_clarity_score(prompt),
        "output_format":        float(has_output_format(prompt)),
        "few_shot_examples":    float(has_few_shot_examples(prompt)),
        "constraints":          float(has_constraints(prompt)),
        "context_completeness": float(has_context(prompt)),
        "task_decomposition":   float(has_task_decomposition(prompt)),
    }
    total = sum(raw[k] * WEIGHTS[k] for k in WEIGHTS)
    return {"dimensions": raw, "total": round(total, 3), "max_possible": 1.0}

## Step 2: Score Example Prompts
We compare a minimal prompt against a well-structured prompt.

In [ ]:
# Weak prompt - minimal structure
weak_prompt = "Summarize this text."

# Strong prompt - all dimensions covered
strong_prompt = """You are an expert summarizer.
Given the following article:
<context>
{text}
</context>
Summarize it in 3 bullet points. Do not include opinions. Respond in plain text.

Example 1: Input article about climate → Output: 3 bullet points about key facts.
"""

# Additional example prompts to demonstrate range
prompts_to_score = [
    ("weak_bare", "What is the capital of France?"),
    ("weak_no_format", "Explain machine learning to me."),
    ("medium_role_only", "You are a Python expert. Explain list comprehensions."),
    ("medium_format_only", "List the top 5 sorting algorithms in JSON format."),
    ("strong_full", strong_prompt),
]

print("=== Prompt Quality Scores ===\n")
for name, p in prompts_to_score:
    result = score_prompt(p)
    print(f"{name}: total={result['total']}")
    for dim, val in result['dimensions'].items():
        print(f"  {dim}: {val:.2f}")
    print()

print("Weak prompt score:", score_prompt(weak_prompt))
print("Strong prompt score:", score_prompt(strong_prompt))

## Step 3: Improve a Low-Scoring Prompt with Claude
We use the score report to identify missing dimensions, then ask Claude
to rewrite the prompt with those dimensions filled in.

In [ ]:
def build_improvement_request(prompt: str, scores: dict) -> str:
    missing = [k for k, v in scores["dimensions"].items() if v < 0.5]
    return f"""You are a prompt engineering expert.
The following prompt is missing these quality dimensions: {missing}.
Rewrite it to include all missing dimensions while preserving the original intent.

Original prompt:
{prompt}

Return only the improved prompt, no explanation."""

result = client.messages.create(
    model=MODEL_NAME,
    max_tokens=512,
    messages=[{"role": "user", "content": build_improvement_request(weak_prompt, score_prompt(weak_prompt))}],
)
improved = result.content[0].text
print("Improved prompt:")
print(improved)
print("\nImproved score:", score_prompt(improved))

## Step 4: Batch Score Multiple Prompts
Score a list of prompts and display a summary table.

In [ ]:
all_prompts = [weak_prompt, strong_prompt, improved]
rows = []
for p in all_prompts:
    s = score_prompt(p)
    rows.append({"prompt_preview": p[:60] + "...", "total": s["total"], **s["dimensions"]})

print("=== Batch Score Summary ===")
for row in rows:
    print(f"\nPrompt: {row['prompt_preview']}")
    print(f"  Total score: {row['total']}")
    for k, v in row.items():
        if k not in ("prompt_preview", "total"):
            print(f"  {k}: {v:.2f}")